# 02 - Conversão dos CSVs para Delta Lake

Lemos os cinco CSVs do bucket landing-zone diretamente pelo Spark
e gravamos os dados no formato Delta Lake no bucket bronze.

Os CSVs permanecem no landing-zone. Cada tabela Delta recebe
seu próprio caminho no bronze, contendo dados Parquet e _delta_log.

## 1. Configuração do MinIO e do Spark

Carregamos o suporte ao Delta Lake e o conector Hadoop AWS.
O protocolo s3a permite ao Spark acessar o MinIO.

Usamos dois núcleos para este exemplo pequeno. As credenciais
são carregadas do arquivo .env.

In [1]:
import os
from pathlib import Path
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

raiz = Path.cwd()
if not (raiz / "pyproject.toml").is_file():
    raiz = raiz.parent

load_dotenv(raiz / ".env", override=True)

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name="us-east-1",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)

spark = (
    SparkSession.builder
    .appName("Loja - CSV para Delta no MinIO")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.2.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.databricks.delta.snapshotPartitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Versão do Spark:", spark.version)

26/10/13 19:31:45 WARN Utils: Your hostname, YasminNitroV15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/13 19:31:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/bfontanayasmin/.ivy2/cache
The jars for the packages stored in: /home/bfontanayasmin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4510fe25-8028-4ef0-87a5-84506eacc9af;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/bfontanayasmin/projetos/trabalhoBancodeDados2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


	found org.apache.hadoop#hadoop-aws;3.3.4 in central


	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central


...

	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (199ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...


	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (9153ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-openssl-1.0.7.Final.jar ...
	[SUCCESSFUL ] org.wildfly.openssl#wildfly-openssl;1.0.7.Final!wildfly-openssl.jar (117ms)
:: resolution report :: resolve 4995ms :: artifacts dl 9479ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------

	3 artifacts copied, 3 already retrieved (275421kB/173ms)
26/10/13 19:32:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Versão do Spark: 3.5.3


## 2. Preparar o bronze e listar os arquivos de origem

Criamos o bucket bronze quando necessário e listamos os CSVs
existentes no landing-zone.

In [2]:
try:
    s3.head_bucket(Bucket="bronze")
except ClientError as erro:
    status = erro.response["ResponseMetadata"]["HTTPStatusCode"]
    if status == 404:
        s3.create_bucket(Bucket="bronze")
    else:
        raise

arquivos = []
paginas = s3.get_paginator("list_objects_v2").paginate(
    Bucket="landing-zone"
)
for pagina in paginas:
    for objeto in pagina.get("Contents", []):
        if objeto["Key"].endswith(".csv"):
            arquivos.append(objeto["Key"])

arquivos.sort()
assert len(arquivos) >= 5, "Esperados pelo menos cinco CSVs."

for arquivo in arquivos:
    print("Origem:", arquivo)
print("Bucket bronze preparado.")

Origem: categorias.csv
Origem: clientes.csv
Origem: itens_pedido.csv
Origem: pedidos.csv
Origem: produtos.csv
Bucket bronze preparado.


## 3. Definir os tipos dos dados

CSV não registra os tipos das colunas. Por isso definimos os schemas
conforme as tabelas do SQL Server, preservando inteiros, datas e
valores monetários DECIMAL(10,2).

Os nomes e tipos são preservados nesta conversão. As restrições de
chave primária e estrangeira do SQL Server não são transferidas.

In [3]:
schemas = {
    "categorias": (
        "id_categoria INT, nome_categoria STRING"
    ),
    "produtos": (
        "id_produto INT, nome_produto STRING, "
        "preco DECIMAL(10,2), estoque INT, id_categoria INT"
    ),
    "clientes": (
        "id_cliente INT, nome_cliente STRING, email STRING"
    ),
    "pedidos": (
        "id_pedido INT, id_cliente INT, "
        "data_pedido DATE, status_pedido STRING"
    ),
    "itens_pedido": (
        "id_item INT, id_pedido INT, id_produto INT, "
        "quantidade INT, preco_unitario DECIMAL(10,2)"
    )
}

tabelas_origem = {arquivo.removesuffix(".csv") for arquivo in arquivos}
assert tabelas_origem == set(schemas), "Confira os schemas dos CSVs."
print("Schemas definidos para as cinco tabelas.")

Schemas definidos para as cinco tabelas.


## 4. Ler os CSVs e gravar em Delta

O Spark lê cada arquivo diretamente do MinIO.
format("delta") define o formato de saída.

overwrite permite repetir a conversão, substituindo os dados atuais
da tabela Delta pelos registros extraídos da origem.

In [4]:
quantidades = {}

for arquivo in arquivos:
    tabela = arquivo.removesuffix(".csv")
    origem = f"s3a://landing-zone/{arquivo}"
    destino = f"s3a://bronze/{tabela}"

    dados = (
        spark.read
        .schema(schemas[tabela])
        .option("header", "true")
        .option("mode", "FAILFAST")
        .option("dateFormat", "yyyy-MM-dd")
        .csv(origem)
    )

    quantidade = dados.count()
    assert quantidade > 0, f"Tabela vazia: {tabela}"

    (
        dados.write
        .format("delta")
        .mode("overwrite")
        .save(destino)
    )

    quantidades[tabela] = quantidade
    print(f"Convertida: {tabela} — {quantidade} registros")

26/10/13 19:32:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Convertida: categorias — 2 registros


Convertida: clientes — 5 registros


Convertida: itens_pedido — 10 registros


Convertida: pedidos — 5 registros


Convertida: produtos — 10 registros


## 5. Validar as tabelas Delta

Conferimos se cada destino é uma tabela Delta válida, se mantém
as quantidades e se preserva os tipos definidos.

Ao terminar, encerramos a sessão Spark.

In [5]:
for tabela, quantidade in quantidades.items():
    caminho = f"s3a://bronze/{tabela}"
    assert DeltaTable.isDeltaTable(spark, caminho)

    dados_delta = spark.read.format("delta").load(caminho)
    assert dados_delta.count() == quantidade

    schema_esperado = spark.read.schema(schemas[tabela]).csv(
        f"s3a://landing-zone/{tabela}.csv",
        header=True
    ).schema
    assert dados_delta.schema == schema_esperado

    print(f"Validado: {tabela} — {quantidade} registros — formato Delta")

print(f"Conversão concluída: {len(quantidades)} tabelas no bucket bronze.")
print(f"Total: {sum(quantidades.values())} registros.")

spark.stop()
print("Sessão Spark encerrada.")

26/10/13 19:32:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Validado: categorias — 2 registros — formato Delta


Validado: clientes — 5 registros — formato Delta


Validado: itens_pedido — 10 registros — formato Delta


Validado: pedidos — 5 registros — formato Delta


Validado: produtos — 10 registros — formato Delta
Conversão concluída: 5 tabelas no bucket bronze.
Total: 32 registros.
Sessão Spark encerrada.
